### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ieee_fraud_detection",
    dataset_year="2019",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/ieee-fraud-detection/overview",
    download_description=r"""
kaggle competitions download -c ieee-fraud-detection -p local-data-warehouse/ieee_fraud_detection \
&& unzip local-data-warehouse/ieee_fraud_detection/ieee-fraud-detection.zip -d local-data-warehouse/ieee_fraud_detection/ \
&& rm local-data-warehouse/ieee_fraud_detection/ieee-fraud-detection.zip
""",
    # References
    academic_reference_bibtex=r"""@misc{ieee-fraud-detection,
    author = {Addison Howard and Bernadette Bouchon-Meunier and IEEE CIS and inversion and John Lei and Lynn@Vesta and Marcus2010 and Prof. Hussein Abbass},
    title = {IEEE-CIS Fraud Detection},
    year = {2019},
    howpublished = {\url{https://kaggle.com/competitions/ieee-fraud-detection}},
    note = {Kaggle}
}
""",
    academic_reference_bibtex_key="ieee-fraud-detection",
    license="non-commercial purposes only, including academic research",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
- We use insights of the first place solution on Kaggle for conceptualizing the task: https://www.kaggle.com/competitions/ieee-fraud-detection/discussion/111284
- The data is given as transactions, but the task is to predict fraudulent clients. Once a transaction is detected as fraud, the entire account is considered fraudulent.
- The competition host commented on the labeling logic: "The logic of our labeling is define reported chargeback on the card as fraud transaction (isFraud=1) and transactions posterior to it with either user account, email address or billing address directly linked to these attributes as fraud too" (https://www.kaggle.com/c/ieee-fraud-detection/discussion/101203#589276)
- We load train_transaction.csv and train_identity.csv and merge them on TransactionID.
- We derive Transaction_date from TransactionDT using the competition reference start date 2017-11-30.
- The Kaggle competition also provides test_transaction.csv and test_identity.csv, but they are unlabeled and are therefore excluded from df for curation checks.
- We normalize the D-columns, since they correspond to days since a certain event, we can normalize them by the transaction date to get the actual day of the event. This is also what the first place solution on Kaggle did.
- We add a uid feature used by the first place solution on Kaggle to identify unique users. Note that this feature is not perfect and just an approximation by the Kaggle users. Moreover, due to missing values, for ~15% of the samples no uid could be reconstructed.
- Following the first place Kaggle solution, which used month-based cross-validation, we create splits using forecasting horizons of 1 month.
- The competition test data left a planning window of one month, so do we.
- Anomaly: Missing values.
- Anomaly: Several features might allow to approximate to the uid and can lead to overfitting.
- Anomaly: Grouped data with members of a group often sharing the same label, depending on whether fraud occurred previously or not.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="isFraud",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    # For classification
    stratify_on="isFraud",
    # For time
    time_on="Transaction_date",
)

## Preprocessing

In [ ]:
import pandas as pd
import datetime
import numpy as np

# LOAD TRAIN
df = pd.read_csv(dataset_mold.path / 'train_transaction.csv', index_col='TransactionID', engine="pyarrow")
train_id = pd.read_csv(dataset_mold.path / 'train_identity.csv', index_col='TransactionID', engine="pyarrow")
df = df.merge(train_id, how='left', left_index=True, right_index=True)

# LOAD TEST
# X_test = pd.read_csv(dataset_mold.path / 'test_transaction.csv', index_col='TransactionID', engine="pyarrow")
# test_id = pd.read_csv(dataset_mold.path / 'test_identity.csv', index_col='TransactionID', engine="pyarrow")
# fix = {o:n for o, n in zip(test_id.columns, train_id.columns)}
# test_id.rename(columns=fix, inplace=True)
# df_test = X_test.merge(test_id, how='left', left_index=True, right_index=True)

START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
df['Transaction_date'] = df['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds=x)))
# df['DT_M'] = (df['Transaction_date'].dt.year - 2017) * 12 + df['Transaction_date'].dt.month

df['day'] = df['TransactionDT'] / (24 * 60 * 60)

# NORMALIZE D COLUMNS, since they correspond to days since a certain event, we can normalize them by the transaction date to get the actual day of the event. This is also what the first place solution on Kaggle did.
for i in range(1,16):
    if i in [1,2,3,5,9]: continue
    df['D'+str(i)] =  df['D'+str(i)] - df['TransactionDT']/np.float32(24*60*60)

df = df.drop(columns=['TransactionDT'])

# This is the uid that as used for feature engineering in the first place solution on Kaggle. They also have a better way to find uids 
# df['uid'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str)  + '_' + np.floor(df.day - df.D1).astype(str)
# df.loc[np.logical_or(df['D1'].isna(), df['addr1'].isna()),["uid"]] = np.nan
# Use the uid version that has the best missing values (15%) vs. accuracy for same uid (99.96%)
uids = pd.read_csv('uids_v1_no_multiuid_cleaning.csv',usecols=['TransactionID','uid'],engine="pyarrow")
df = df.merge(uids,on='TransactionID',how='left')

# Obtain correct cat indices
cat_cols = ['ProductCD',"addr1","addr2","P_emaildomain","R_emaildomain","DeviceType","DeviceInfo"]
cat_cols += [f"card{i}" for i in range(1,7)]
cat_cols += [f"M{i}" for i in range(1,10)]
cat_cols += [f"id_{i}" for i in range(12,39)]
cat_cols += ["uid"]

for col in cat_cols:
    df[col] = df[col].astype('category')

df["isFraud"] = df["isFraud"].astype("category")
df = df.sort_values(by=["Transaction_date"]).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata

date_col = task_mold.time_on
target_col = task_mold.target_column_name

months = (df['Transaction_date'].dt.year - 2017) * 12 + df['Transaction_date'].dt.month

# -- For Temporal non-IID data -> manual processing required
splits = {
    0: {
        0: ((months[months<=15]).index.to_list(), (months[months==17]).index.to_list()),
    },
    1: {
        0: ((months[months<=14]).index.to_list(), (months[months==16]).index.to_list()),
    },
    2: {
        0: ((months[months<=13]).index.to_list(), (months[months==15]).index.to_list()),
    },
}

used_in_train = set()
used_in_test = set()
used_data = set()
for split in splits:
    train_idx, test_idx = splits[split][0]
    used_in_train.update(train_idx)
    used_in_test.update(test_idx)
    used_data.update(train_idx)
    used_data.update(test_idx)

    print(f"\n=== Step {split} ===")
    print("Train size:", len(train_idx), "| Test size:", len(test_idx))
    print("Train target mean:", df.loc[train_idx, target_col].astype(int).mean())
    print("Test target mean:", df.loc[test_idx, target_col].astype(int).mean())
    print("% test samples with reoccuring, non-missing uids:", df["uid"].dropna().isin(set(df.loc[test_idx, "uid"]).intersection(set(df.loc[train_idx, "uid"]))).mean())
    print("% test samples with missing uids:", df["uid"].isna().mean())

    assert len(set(train_idx).intersection(set(test_idx)))==0, "Train and test indices overlap!"

print(f"{len(used_data)/df.shape[0]:.4f} of the samples are used.")
print(f"{len(used_in_train)/df.shape[0]:.4f} of the samples are used in training")
print(f"{len(used_in_test)/df.shape[0]:.4f} of the samples are used in testing.")

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="To define three train/test splits, we use the last three months as test sets, leave 1 month as the planning gap, and use the remaining training data.",
    splits=splits,
    # For time data
    time_horizon="1",
    time_horizon_unit="months",
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()